In [28]:
import sagemaker

In [29]:
from sagemaker.huggingface import HuggingFace

In [30]:
role = sagemaker.get_execution_role()

In [31]:
role

'arn:aws:iam::385046010545:role/SageMakerLLMRole'

In [32]:
hyperparameters = {
    "model_id": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    "epochs": 2,
    "per_device_train_batch_size": 2,
    "lr": 2e-5
}

In [39]:
estimator = HuggingFace(
    entry_point="train.py",
    source_dir="./scripts",
    role=role,
    transformers_version="4.36",
    pytorch_version="2.1",
    py_version="py310",
    instance_type="ml.g5.xlarge",
    instance_count=1,
    output_path="s3://llm-model-artifacts-santosh/models/",
    hyperparameters=hyperparameters
)

In [40]:
estimator.fit({
    "train": "s3://llm-finetune-dataset-santosh/datasets/"
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2025-11-15-11-46-24-681


2025-11-15 11:46:25 Starting - Starting the training job
2025-11-15 11:46:25 Pending - Training job waiting for capacity...
2025-11-15 11:46:56 Pending - Preparing the instances for training...
2025-11-15 11:47:24 Downloading - Downloading input data...
2025-11-15 11:47:49 Downloading - Downloading the training image........................
2025-11-15 11:51:37 Training - Training image download completed. Training in progress.bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be

In [42]:
estimator.model_data

's3://llm-model-artifacts-santosh/models/huggingface-pytorch-training-2025-11-15-11-46-24-681/output/model.tar.gz'

In [43]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge"
)

predictor.predict({"inputs": "Explain AWS S3"})


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <cell line: 1>:1                                                                              │
│                                                                                                  │
│ ❱ 1 predictor = estimator.deploy(                                                                │
│   2 │   initial_instance_count=1,                                                                │
│   3 │   instance_type="ml.g5.xlarge"                                                             │
│   4 )                                                                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:1771   │
│ in deploy                                                                                        │
│                                                                                                  │
│   1768 │   │   │   inference_tags=format_tags(tags), training_tags=self.tags                     │
│   1769 │   │   )                                                                                 │
│   1770 │   │                                                                                     │
│ ❱ 1771 │   │   return model.deploy(                                                              │
│   1772 │   │   │   instance_type=instance_type,                                                  │
│   1773 │   │   │   initial_instance_count=initial_instance_count,                                │
│   1774 │   │   │   serializer=serializer,                                                        │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/huggingface/model.p │
│ y:326 in deploy                                                                                  │
│                                                                                                  │
│   323 │   │   │   │   inference_tool=inference_tool,                                             │
│   324 │   │   │   )                                                                              │
│   325 │   │                                                                                      │
│ ❱ 326 │   │   return super(HuggingFaceModel, self).deploy(                                       │
│   327 │   │   │   initial_instance_count,                                                        │
│   328 │   │   │   instance_type,                                                                 │
│   329 │   │   │   serializer,                                                                    │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/model.py:1737 in    │
│ deploy                                                                                           │
│                                                                                                  │
│   1734 │   │   │   return None                                                                   │
│   1735 │   │                                                                                     │
│   1736 │   │   else:  # existing single model endpoint path                                      │
│ ❱ 1737 │   │   │   self._create_sagemaker_model(                                                 │
│   1738 │   │   │   │   instance_type=instance_type,                                              │
│   1739 │   │   │   │   accelerator_type=accelerator_type,                                        │
│   1740 │   │   │   │   tags=tags,                          